# 문제 11. 유·무인 재난지원팀의 드론 영상 차량 탐지

**제한시간 30분 · 난이도 최상 목표 · Python3 · Jupyter · CPU · CSV 제출**

## 상황

재난지원 상황실은 드론 촬영 영상에서 도로 주변 차량의 위치를 파악하려고 합니다. 각 이미지에 있는 차량을 경계 상자(bounding box)로 탐지하고 신뢰도와 함께 제출하세요.

실제 공개 민간 드론 영상과 사람이 주석한 정답을 가공한 가상 MUM-T 지원 과제입니다. 실제 군 영상이나 재난 발생 사실을 기록한 데이터라는 뜻은 아닙니다. 사람의 신원·행동은 예측하지 않습니다.

## Data

| 파일 | 내용 |
|---|---|
| data/train.csv | fit 20장·valid 20장의 이미지 ID, 경로, 크기, scene_id, split |
| data/train_boxes.csv | 학습 40장의 차량 경계 상자 정답 |
| data/test.csv | 테스트 40장의 이미지 ID, 경로, 크기, scene_id |
| data/images/ | 원래 해상도의 RGB JPEG 80장 |
| data/ignore_regions.csv | 평가에서 제외하는 영역. 학습·테스트 모두 제공 |
| data/sample_submission.csv | 컬럼만 있는 빈 제출 예시. 제출 행 수는 탐지 개수에 따라 달라짐 |
| models/yolov8n.onnx | 선택적으로 사용할 수 있는 일반 객체 탐지용 사전학습 모델 |
| detector_helper.py | CPU 추론 및 NMS 도우미 |
| evaluation.py | 검증용 AP50 계산 함수. 테스트 정답은 포함하지 않음 |

### 대상과 좌표

- 평가 클래스는 하나, **class_id=0: 차량**입니다. 승용차·밴·버스·트럭을 합친 범주입니다. 오토바이·자전거·삼륜차는 대상이 아닙니다.
- 좌표는 원본 이미지 왼쪽 위를 (0,0)으로 하는 픽셀 단위 `x_min,y_min,x_max,y_max`입니다. 오른쪽·아래쪽 경계는 이미지 너비·높이까지 허용합니다.
- 원본을 자르거나 크기를 바꿔 추론했다면 제출 전 좌표를 원본 기준으로 복원하세요.
- 같은 scene_id는 같은 원본 촬영 시퀀스의 이미지입니다. fit·valid·test의 scene_id는 겹치지 않습니다. 서로 다른 시퀀스 사이의 지리적 중첩까지 보장하지는 않습니다.
- ignore_regions는 상세 정답이 충분하지 않아 평가에서 제외하는 영역입니다. 탐지 대상 정답으로 사용하지 마세요.

## 초기 코드


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

BASE_DIR = Path(".")
train = pd.read_csv(BASE_DIR / "data/train.csv")
train_boxes = pd.read_csv(BASE_DIR / "data/train_boxes.csv")
test = pd.read_csv(BASE_DIR / "data/test.csv")
ignore_regions = pd.read_csv(BASE_DIR / "data/ignore_regions.csv")
sample_submission = pd.read_csv(BASE_DIR / "data/sample_submission.csv")
display(train.head())
display(Image.open(BASE_DIR / train.iloc[0]["image_path"]))


## 과제

1. 학습 이미지와 정답을 살펴보고 객체 크기·밀집도·가림 등을 확인하세요.
2. 사전학습 탐지 모델로 기본 결과를 얻고 valid에서 AP50을 평가하세요.
3. 해상도·분할 추론·좌표 복원·중복 제거 등 필요한 개선을 선택하고 검증하세요.
4. 테스트 이미지별 차량 위치와 신뢰도를 CSV로 저장하세요.

### 제공 도우미

```python
from detector_helper import Detector, nms
# detector = Detector()  # 로컬 ONNX 모델, CPU 사용
# detections = detector.detect(pil_image, conf=0.03, iou=0.5)
```

`detect`는 `(N,6)` 배열을 반환합니다. 열 순서는 `[x_min,y_min,x_max,y_max,confidence,coco_class_id]`이고, 좌표는 **함수에 넣은 이미지 또는 잘라낸 이미지 기준**입니다. 빈 결과는 `(0,6)`입니다. 내부 크기 조정·패딩의 역변환까지 처리합니다.

제공 모델의 COCO 클래스 2(car)·5(bus)·7(truck)를 기본으로 반환합니다. 모델의 클래스 번호와 **제출 클래스 0**을 구분하세요. 제공 모델에는 밴만의 독립 클래스가 없습니다. 세 클래스의 탐지를 차량 범주로 통합할 수 있지만, 탐지 정확도는 검증해야 합니다.

`nms(boxes,scores,iou_threshold=0.5)`는 겹친 상자를 억제한 뒤 남길 행 인덱스를 신뢰도 순서로 반환합니다. 여러 조각의 결과 결합과 원본 좌표 복원은 직접 구현하세요. 모델을 새로 학습할 의무는 없으며, **다른 사전학습 가중치나 미세조정도 허용**합니다. 제공 모델은 CPU에서 빠르게 시작하기 위한 선택지입니다. 테스트 원본 정답을 외부에서 찾아 쓰거나 평가용 이미지를 학습한 것으로 확인된 가중치를 이용하는 것은 금지합니다.

### 검증과 평가

```python
from evaluation import evaluate
# valid_images = train[train["split"] == "valid"]
# result = evaluate(valid_predictions, train_boxes, valid_images, ignore_regions)
# print(result)
```

**AP@IoU 0.5(이하 AP50), 높을수록 좋음.** 한 클래스이므로 클래스 평균을 별도로 취하지 않습니다.

1. 모든 평가 이미지의 탐지를 신뢰도 내림차순으로 정렬합니다. 동점은 제출 행 순서를 유지합니다.
2. 같은 이미지에서 아직 대응되지 않은 정답과 IoU가0.5 이상이면 가장 많이 겹치는 정답 하나에 대응시킵니다.
3. 정답에 대응되지 않고, 제외 영역과의 교집합이 **탐지 상자 면적의50% 이상**이면 평가에서 무시합니다. 이 비율은 IoU가 아닙니다.
4. 나머지는 오탐입니다. 대응되지 않은 정답은 미탐입니다. 제외 영역보다 정답 대응을 우선합니다.
5. 재현율0~1을0.01 간격으로 잡은101개 지점에서 보간 정밀도를 평균합니다. 해당 재현율에 도달하지 못하면 그 지점은0입니다.

이 과제의 구체적 정의는 제공 평가 함수와 같습니다. 원본 대회의 모든 평가 규칙을 재현한 점수는 아닙니다. `score_100=100*AP50`이며 공식 합격선은 없습니다. 검증은 valid에 하고 테스트 정답으로 설정을 조정하지 마세요.

## 결과 저장

Notebook 옆에 **submission.csv**를 저장하세요.

| image_id | class_id | confidence | x_min | y_min | x_max | y_max |
|---|---:|---:|---:|---:|---:|---:|
| 예시 ID | 0 | 0.82 | 120 | 80 | 165 | 125 |

- 컬럼 순서가 위와 같아야 합니다. 값은 예시이며 실제 정답이 아닙니다.
- 이미지당 최대300개, 객체당 한 행. 탐지가 없는 이미지는 행을 쓰지 않습니다. 전체가 빈 경우에도 헤더는 저장하세요.
- image_id는 test.csv에 있는 값. class_id는0, confidence는0~1의 유한한 실수입니다.
- `0≤x_min<x_max≤width`, `0≤y_min<y_max≤height`. 소수 좌표를 허용합니다.
- 완전히 동일한 탐지 행의 중복·결측·무한대·추가 컬럼은 금지합니다.
- `index=False`로 저장하고 코드와 실행 결과도 Notebook에 저장하세요. 별도 보고서는 없습니다.


In [ ]:
# 여기에 코드를 작성하세요.


## CSV 저장 예시

In [ ]:
# result: 위에서 계산한 탐지 결과 DataFrame
# result.to_csv(BASE_DIR / "submission.csv", index=False, encoding="utf-8-sig")


## 파일 형식 확인

In [ ]:
from evaluation import validate
path = BASE_DIR / "submission.csv"
if path.exists():
    result = pd.read_csv(path)
    validate(result, test)
    print("제출 형식 확인 완료. 점수 채점이나 최종 제출은 아닙니다.")
else:
    print("풀이 후 submission.csv를 생성하세요.")
